# Célula 1 – Imports e transformações

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# --- Constantes e Configurações Globais ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR = "../../data/final"
REGIONS = ["forehead", "chin", "nose", "left_cheek", "right_cheek"]
region_to_idx = {r: i for i, r in enumerate(REGIONS)}
BATCH_SIZE = 16
NUM_EPOCHS = 40

# --- Transformações para Aumento de Dados e Normalização ---

# Aumento de dados robusto para o conjunto de treino
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 2.0))], p=0.2),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)), # Regularização adicional
])

# Transformação padrão para validação/teste (sem aumento de dados)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Transformação para teste de robustez, alinhada com as augmentations de treino
robust_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 2.0))], p=0.3),
    transforms.ToTensor()
])

print(f"Dispositivo utilizado: {device}")

# Célula 2 – Dataset customizado

In [ ]:
class AcneDataset(Dataset):
    """Classe de Dataset genérica para carregar imagens de treino, val ou teste."""
    def __init__(self, base_dir, splits, transform=None, include_val_in_train=False):
        self.samples = []
        self.transform = transform
        
        # Se include_val_in_train for True, adiciona 'val' à lista de splits
        if include_val_in_train and 'train' in splits:
            splits.append('val')

        for region in REGIONS:
            for split in splits:
                split_dir = os.path.join(base_dir, region, split)
                if os.path.isdir(split_dir):
                    # Usando ImageFolder para encontrar imagens e seus rótulos
                    ds = datasets.ImageFolder(split_dir, transform=None)
                    for path, label in ds.samples:
                        self.samples.append((path, label, region))

        print(f"Carregadas {len(self.samples)} imagens dos splits: {', '.join(splits)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, region = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        region_idx = region_to_idx[region]
        return img, label, region_idx

# --- Instanciando os Datasets ---
# Dataset de treino combinado (treino + validação)
train_ds = AcneDataset(BASE_DIR, splits=['train'], transform=train_transform, include_val_in_train=True)

# Dataset de teste padrão
test_ds = AcneDataset(BASE_DIR, splits=['test'], transform=val_transform)

# Dataset de teste para validação de robustez durante o treino
test_robust_ds = AcneDataset(BASE_DIR, splits=['test'], transform=robust_transform)

# --- Criando os DataLoaders ---
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
test_robust_loader = DataLoader(test_robust_ds, batch_size=BATCH_SIZE, shuffle=False)

# Célula 3 – Modelo

In [ ]:
class OptimizedMultiRegionResNet18(nn.Module):
    def __init__(self, num_classes=4, num_regions=5):
        super().__init__()
        # Carrega a arquitetura sem pesos, pois vamos carregar os nossos
        self.resnet = models.resnet18(weights=None)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()
        
        self.region_embed = nn.Embedding(num_regions, 64)
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features + 64, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )

    def forward(self, x, region_idx):
        img_features = self.resnet(x)
        # CORREÇÃO: Garante que o índice é Long e remove a dimensão extra
        region_features = self.region_embed(region_idx.long()).squeeze(1)
        combined = torch.cat([img_features, region_features], dim=1)
        return self.classifier(combined)

print("Definição do modelo corrigida e pronta.")

# Célula 4 – Focal Loss

In [ ]:
def evaluate_model(model, loader, device):
    """Função auxiliar para avaliar a acurácia do modelo em um dado DataLoader."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels, regions in loader:
            imgs, labels, regions = imgs.to(device), labels.to(device), regions.to(device)
            outputs = model(imgs, regions)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / total

def train_model(model, train_loader, test_loader, robust_loader, criterion, optimizer, scheduler, num_epochs):
    best_robust_acc = 0.0
    best_test_acc = 0.0
    patience = 10
    patience_counter = 0
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for imgs, labels, regions in train_loader:
            imgs, labels, regions = imgs.to(device), labels.to(device), regions.to(device)
            
            optimizer.zero_grad()
            outputs = model(imgs, regions)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()

        epoch_train_loss = running_loss / len(train_loader)
        
        # Avalia a cada 5 épocas para economizar tempo
        if (epoch + 1) % 5 == 0 or epoch == num_epochs - 1:
            epoch_test_acc = evaluate_model(model, test_loader, device)
            epoch_robust_acc = evaluate_model(model, robust_loader, device)
            
            print(f"Epoch {epoch+1}/{num_epochs} | "
                  f"Loss: {epoch_train_loss:.4f} | "
                  f"Test Acc: {epoch_test_acc:.2f}% | "
                  f"Robust Acc: {epoch_robust_acc:.2f}%")
            
            scheduler.step(epoch_test_acc) # Atualiza o learning rate com base na acurácia de teste
            
            gap = epoch_test_acc - epoch_robust_acc
            
            # Critério para salvar o melhor modelo: alta acurácia de robustez com um gap controlado
            if epoch_robust_acc > best_robust_acc and gap < 18:
                best_robust_acc = epoch_robust_acc
                best_test_acc = epoch_test_acc
                patience_counter = 0
                torch.save(model.state_dict(), "best_robust_model.pth")
                print(f"  ✅ Modelo salvo! Robusto: {best_robust_acc:.2f}%, Gap: {gap:.1f}%")
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}. Melhor acurácia robusta: {best_robust_acc:.2f}%")
                break
    
    print("\n=== TREINAMENTO FINALIZADO ===")
    print(f"Melhor Acurácia de Teste: {best_test_acc:.2f}%")
    print(f"Melhor Acurácia Robusta: {best_robust_acc:.2f}%")
    
    return best_test_acc, best_robust_acc

# Célula 5 – Preparar DataLoaders e Loss

In [ ]:
# --- Cross-Entropy com Pesos para Lidar com Desbalanceamento de Classes ---
all_labels = [label for _, label, _ in train_ds]
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(all_labels), y=all_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# Scheduler para ajustar a taxa de aprendizado dinamicamente
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=True
)

# --- Iniciar o Treinamento ---
best_test, best_robust = train_model(
    model, train_loader, test_loader, test_robust_loader,
    criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS
)

# Célula 6 – Métricas


In [ ]:
print("--- Iniciando Análise de Performance Principal ---")
# Carrega o modelo para garantir que estamos usando a versão correta
model_to_test = OptimizedMultiRegionResNet18().to(device)
model_to_test.load_state_dict(torch.load("best_robust_model.pth"))
model_to_test.eval()

# Coleta as previsões
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels, regions in test_loader:
        imgs, labels, regions = imgs.to(device), labels.to(device), regions.to(device)
        outputs = model_to_test(imgs, regions)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Gera o relatório
print("\n" + "="*50)
print("RELATÓRIO DE CLASSIFICAÇÃO DETALHADO")
print("="*50)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# --- CÉLULA DE ANÁLISE COMPLETA E AUTOSSUFICIENTE ---

print("Iniciando análise de estabilidade com ambiente controlado...")

# 1. Imports necessários para esta célula
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm
from PIL import Image

# 2. Definições essenciais para garantir que não há dependências externas
# (Estes valores devem ser os mesmos da sua Célula 1)
CLASS_NAMES = ['Grau 1', 'Grau 2', 'Grau 3', 'Grau 4'] 
NUM_RUNS = 30 

# 3. Definição da transformação de validação CORRETA
# Esta é a parte crítica: garantimos que o pré-processamento está aqui.
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 4. Criação de um DataLoader de teste local para esta célula
# Isso garante que estamos usando 100% os dados corretos.
print("Criando um DataLoader de teste seguro com as transformações corretas...")
# (A classe AcneDataset precisa ter sido definida na Célula 2)
test_ds_safe = AcneDataset(BASE_DIR, splits=['test'], transform=val_transform)
test_loader_safe = DataLoader(test_ds_safe, batch_size=16, shuffle=False)
print("DataLoader seguro criado.")

# 5. Carregamento do modelo
model_to_test = OptimizedMultiRegionResNet18()
model_to_test.load_state_dict(torch.load("best_robust_model.pth"))
model_to_test.to(device)
model_to_test.eval()

# 6. Definição da transformação aleatória para a análise
eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor()
])

# 7. Função de análise (a mesma de antes)
def run_full_metric_analysis(model, loader, num_runs):
    results_history = {name: {'precision': [], 'recall': [], 'f1-score': [], 'accuracy': []} for name in CLASS_NAMES}
    accuracy_history = []
    
    for _ in tqdm(range(num_runs), desc="Progresso da Análise"):
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels, regions in loader:
                # O loader já fornece tensores 224x224 normalizados.
                # A transformação aleatória é aplicada sobre eles.
                transformed_imgs = torch.stack([eval_transform(img.cpu()) for img in imgs]).to(device)
                labels, regions = labels.to(device), regions.to(device)
                outputs = model(transformed_imgs, regions)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
        cm = confusion_matrix(all_labels, all_preds)
        accuracy_history.append(report['accuracy'])
        
        for i, name in enumerate(CLASS_NAMES):
            if name in report:
                results_history[name]['precision'].append(report[name]['precision'])
                results_history[name]['recall'].append(report[name]['recall'])
                results_history[name]['f1-score'].append(report[name]['f1-score'])
                tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp; tn = cm.sum() - (tp + fp + fn)
                class_accuracy = (tp + tn) / cm.sum()
                results_history[name]['accuracy'].append(class_accuracy)

    final_metrics = {}
    for name in CLASS_NAMES:
        final_metrics[name] = {
            'precision_mean': np.mean(results_history[name]['precision']), 'precision_std': np.std(results_history[name]['precision']),
            'recall_mean': np.mean(results_history[name]['recall']), 'recall_std': np.std(results_history[name]['recall']),
            'f1-score_mean': np.mean(results_history[name]['f1-score']), 'f1-score_std': np.std(results_history[name]['f1-score']),
            'accuracy_mean': np.mean(results_history[name]['accuracy']), 'accuracy_std': np.std(results_history[name]['accuracy']),
        }
    return final_metrics, accuracy_history

# 8. Execução da análise usando o DataLoader seguro
final_metrics_by_class, acc_history = run_full_metric_analysis(model_to_test, test_loader_safe, NUM_RUNS)
mean_accuracy = np.mean(acc_history)
std_accuracy = np.std(acc_history)

# 9. Geração da tabela final
print("\n\n" + "="*90)
print("TABELA DE PERFORMANCE DO MODELO (ANÁLISE SEGURA)")
print("="*90)
print("| Métrica | Classe | Precision | Recall | F1-Score | Acurácia (por Classe) (%) |")
print("| :--- | :--- | :--- | :--- | :--- | :--- |")
for i, name in enumerate(CLASS_NAMES):
    metrics = final_metrics_by_class[name]
    metric_group = "**Por Classe**" if i == 0 else ""
    precision_str = f"{metrics['precision_mean']:.2f} ± {metrics['precision_std']:.2f}"
    recall_str = f"{metrics['recall_mean']:.2f} ± {metrics['recall_std']:.2f}"
    f1_str = f"{metrics['f1-score_mean']:.2f} ± {metrics['f1-score_std']:.2f}"
    acc_class_str = f"{(metrics['accuracy_mean'] * 100):.2f} ± {(metrics['accuracy_std'] * 100):.2f}"
    print(f"| {metric_group} | {name} ({i}) | {precision_str} | {recall_str} | {f1_str} | {acc_class_str} |")
print("| **---** | **---** | **---** | **---** | **---** | **---** |")
acc_geral_str = f"{(mean_accuracy * 100):.2f} ± {(std_accuracy * 100):.2f}"
print(f"| **Geral** | **Acurácia** | - | - | - | **{acc_geral_str}** |")

In [ ]:
# Obter previsões e rótulos se ainda não tivermos
if 'preds' not in locals():
    preds, labels = get_all_preds_labels(model, test_loader)

# Encontrar os índices das imagens classificadas incorretamente
misclassified_indices = np.where(preds != labels)[0]
print(f"Total de erros no conjunto de teste: {len(misclassified_indices)} de {len(labels)} imagens.")

# Selecionar alguns exemplos aleatórios para exibir
num_examples_to_show = min(len(misclassified_indices), 10)
random_indices = np.random.choice(misclassified_indices, num_examples_to_show, replace=False)

# O test_ds usa val_transform que converte para Tensor. Precisamos inverter para visualizar.
def tensor_to_pil(tensor):
    # Clona o tensor para não modificar o original, move para CPU
    img = tensor.clone().detach().cpu()
    # Desnormaliza se houver normalização (neste caso não há, mas é uma boa prática)
    # img = img * std + mean 
    img = transforms.ToPILImage()(img)
    return img

# --- Plotar os exemplos ---
fig, axes = plt.subplots(2, 5, figsize=(20, 9))
fig.suptitle('Exemplos de Imagens Classificadas Incorretamente', fontsize=18)
axes = axes.flatten()

for i, img_idx in enumerate(random_indices):
    # test_ds não tem as transformações aleatórias, então podemos pegar a imagem original
    img_tensor, true_label_idx, _ = test_ds[img_idx]
    
    img_pil = tensor_to_pil(img_tensor)
    
    pred_label_idx = preds[img_idx]
    
    true_class_name = CLASS_NAMES[true_label_idx]
    pred_class_name = CLASS_NAMES[pred_label_idx]
    
    ax = axes[i]
    ax.imshow(img_pil)
    ax.set_title(f"Verdadeiro: {true_class_name}\nPrevisto: {pred_class_name}", 
                 color="red", fontsize=12)
    ax.axis('off')

# Esconder eixos não utilizados
for i in range(num_examples_to_show, len(axes)):
    axes[i].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import torch.nn.functional as F

# --- PARÂMETROS ---
NUM_STABILITY_RUNS = 30

# Define a transformação aleatória para ser usada em cada execução do teste
stability_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
])

def run_stability_test(model, loader, num_runs):
    # (A função run_stability_test permanece a mesma)
    accuracies = []
    print(f"Iniciando teste de estabilidade com {num_runs} execuções...")
    for i in range(num_runs):
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels, region_indices in loader:
                transformed_imgs = torch.stack([stability_transform(img) for img in imgs]).to(device)
                labels, region_indices = labels.to(device), region_indices.to(device)
                outputs = model(transformed_imgs, region_indices)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        run_accuracy = 100 * correct / total
        accuracies.append(run_accuracy)
        print(f"  Execução {i+1}/{num_runs}, Acurácia: {run_accuracy:.2f}%")
    return accuracies

# --- CARREGAMENTO DO MODELO SALVO ---
print("Carregando o melhor modelo salvo de 'best_robust_model.pth'...")
# Primeiro, instanciamos a arquitetura do modelo
model_to_test = OptimizedMultiRegionResNet18()
# Em seguida, carregamos os pesos salvos
model_to_test.load_state_dict(torch.load("best_robust_model.pth"))
# Movemos o modelo para o dispositivo (CPU/GPU)
model_to_test.to(device)
# **Importante:** Colocamos o modelo em modo de avaliação
model_to_test.eval()
# ------------------------------------

# Executa o teste com o modelo carregado
accuracy_distribution = run_stability_test(model_to_test, test_loader, NUM_STABILITY_RUNS)

# --- Visualização do Boxplot ---
plt.figure(figsize=(10, 7))
sns.boxplot(y=accuracy_distribution, width=0.3, palette="viridis")
sns.stripplot(y=accuracy_distribution, jitter=True, color="red", alpha=0.7, label="Execuções Individuais")
plt.title('Distribuição da Acurácia do Modelo em Múltiplas Execuções', fontsize=16)
plt.ylabel('Acurácia (%)')
plt.ylim(bottom=min(accuracy_distribution) - 5, top=100)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend()
plt.show()

# --- Estatísticas Descritivas ---
print("\n" + "="*50)
print("ESTATÍSTICAS DE ESTABILIDADE DA ACURÁCIA")
print("="*50)
print(f"Acurácia Média:     {np.mean(accuracy_distribution):.2f}%")
print(f"Desvio Padrão:      {np.std(accuracy_distribution):.2f}%")
print(f"Mediana (Q2):       {np.median(accuracy_distribution):.2f}%")
print(f"Acurácia Mínima:    {np.min(accuracy_distribution):.2f}%")
print(f"Acurácia Máxima:    {np.max(accuracy_distribution):.2f}%")

In [ ]:
def analyze_prediction_confidence(model, loader):
    correct_confidences = []
    incorrect_confidences = []
    with torch.no_grad():
        for imgs, labels, region_indices in loader:
            imgs, labels, region_indices = imgs.to(device), labels.to(device), region_indices.to(device)
            outputs = model(imgs, region_indices)
            probabilities = F.softmax(outputs, dim=1)
            confidences, predicted_classes = torch.max(probabilities, 1)
            for i in range(len(labels)):
                if predicted_classes[i] == labels[i]:
                    correct_confidences.append(confidences[i].item())
                else:
                    incorrect_confidences.append(confidences[i].item())
    return correct_confidences, incorrect_confidences

print("Carregando o melhor modelo salvo de 'best_robust_model.pth'...")
model_to_test = OptimizedMultiRegionResNet18()
model_to_test.load_state_dict(torch.load("best_robust_model.pth"))
model_to_test.to(device)
model_to_test.eval() 

correct_conf, incorrect_conf = analyze_prediction_confidence(model_to_test, test_loader)

# Preparar dados para o Seaborn usando um DataFrame do Pandas
df_correct = pd.DataFrame({'Confiança': correct_conf, 'Resultado': 'Correta'})
df_incorrect = pd.DataFrame({'Confiança': incorrect_conf, 'Resultado': 'Incorreta'})
df_confidence = pd.concat([df_correct, df_incorrect], ignore_index=True)

# --- Visualização do Boxplot ---
plt.figure(figsize=(10, 7))
sns.boxplot(x='Resultado', y='Confiança', data=df_confidence, palette=["skyblue", "salmon"])
plt.title('Distribuição da Confiança do Modelo por Resultado da Previsão', fontsize=16)
plt.ylabel('Confiança da Previsão (Probabilidade Softmax)')
plt.xlabel('Tipo de Previsão')
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("\n" + "="*50)
print("ESTATÍSTICAS DE CONFIANÇA")
print("="*50)
print(f"Confiança Média (Previsões Corretas):   {np.mean(correct_conf):.2%}")
print(f"Confiança Média (Previsões Incorretas): {np.mean(incorrect_conf):.2%}")

In [ ]:
from collections import defaultdict
from sklearn.metrics import classification_report

CLASS_NAMES = ['Grau 1', 'Grau 2', 'Grau 3', 'Grau 4'] 

def detailed_analysis_by_region(model, loader, device):
    """
    Coleta previsões e rótulos para cada região e gera um relatório de 
    classificação detalhado para cada uma.
    """
    # Dicionário para armazenar listas de rótulos verdadeiros e previstos por região
    region_results = defaultdict(lambda: {'y_true': [], 'y_pred': []})
    
    model.eval()
    with torch.no_grad():
        for imgs, labels, region_indices in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            region_indices = region_indices.to(device)
            
            outputs = model(imgs, region_indices)
            _, preds = torch.max(outputs, 1)
            
            for i in range(len(labels)):
                region_name = REGIONS[region_indices[i].item()]
                region_results[region_name]['y_true'].append(labels[i].item())
                region_results[region_name]['y_pred'].append(preds[i].item())
    
    # --- Geração dos Relatórios ---
    print("=" * 70)
    print("RELATÓRIO DE CLASSIFICAÇÃO DETALHADO POR REGIÃO FACIAL")
    print("=" * 70)
    
    for region, results in region_results.items():
        print(f"\n\n--- Análise para a Região: {region.upper()} ---")
        
        if len(results['y_true']) > 0:
            try:
                # Gera e imprime o relatório do scikit-learn
                report = classification_report(
                    results['y_true'], 
                    results['y_pred'], 
                    target_names=CLASS_NAMES,
                    zero_division=0 
                )
                print(report)
            except ValueError as e:
                print(f"Não foi possível gerar o relatório para esta região: {e}")
        else:
            print("Nenhuma amostra de teste encontrada para esta região.")
            
model_to_test = OptimizedMultiRegionResNet18()
model_to_test.load_state_dict(torch.load("best_robust_model.pth"))
model_to_test.to(device)

detailed_analysis_by_region(model_to_test, test_loader, device)

In [ ]:
print("--- Iniciando conversão do modelo para Pytorch Mobile (.ptl) ---")

# 1. Instancie a arquitetura (usando a classe da Célula 3)
model_to_convert = OptimizedMultiRegionResNet18()

# 2. Carregue os pesos que você treinou, mapeando para a CPU
model_to_convert.load_state_dict(torch.load("best_robust_model.pth", map_location=torch.device('cpu')))
model_to_convert.eval()

# 3. Crie exemplos de entrada para o "trace"
example_image = torch.rand(1, 3, 224, 224)
example_region = torch.tensor([[0]]) 

# 4. Execute o trace, otimize e salve
traced_script_module = torch.jit.trace(model_to_convert, (example_image, example_region))
from torch.utils.mobile_optimizer import optimize_for_mobile
optimized_traced_module = optimize_for_mobile(traced_script_module)
optimized_traced_module._save_for_lite_interpreter("acne_model.ptl")

print("\nModelo convertido para acne_model.ptl com sucesso! ✅")